# SQL Fundamentals
### Data Science Core Concepts 101 — by Yunhan Jin

This notebook covers SQL fundamentals using SQLite — no database server required, runs entirely in Python.

---

## Section 1: The Basics

### 1.1 Setup
We use SQLite — built into Python, no server required. The database and sample data are created below and used throughout this notebook.

Two key concepts before looking at the schema:
- **Primary key**: a column that uniquely identifies each row in a table — no duplicates, no NULLs. In `employees`, `id` is the primary key.
- **Foreign key**: a column that references the primary key of another table, creating a link between them. In `employees`, `dept` references `dept` in the `departments` table — this is what makes JOINs possible.

In [1]:
import sqlite3
import pandas as pd

# Create in-memory database
conn = sqlite3.connect(":memory:")
cursor = conn.cursor()

cursor.executescript("""
    CREATE TABLE employees (
        id       INTEGER PRIMARY KEY,
        name     TEXT,
        dept     TEXT,
        salary   INTEGER,
        years    INTEGER
    );

    INSERT INTO employees VALUES
        (1, 'Alice',   'Engineering', 95000, 4),
        (2, 'Bob',     'Marketing',   72000, 2),
        (3, 'Carol',   'Engineering', 110000, 7),
        (4, 'David',   'Marketing',   68000, 1),
        (5, 'Eve',     'Engineering', 102000, 5),
        (6, 'Frank',   'HR',          61000, 3),
        (7, 'Grace',   'HR',          64000, 6),
        (8, 'Hank',    'Marketing',   75000, 4);

    CREATE TABLE departments (
        dept    TEXT PRIMARY KEY,
        manager TEXT,
        budget  INTEGER
    );

    INSERT INTO departments VALUES
        ('Engineering', 'Carol', 500000),
        ('Marketing',   'Hank',  200000),
        ('HR',          'Grace', 150000),
        ('Finance',     'Ivan',  180000);
""")

conn.commit()
print("Database ready.")

Database ready.


### 1.2 SELECT & FROM
`SELECT` specifies which columns to retrieve. `FROM` specifies the table. These are the foundation of every SQL query.

In [2]:
# Select all columns
query = """
SELECT *
FROM employees
"""
pd.read_sql(query, conn)

,id,name,dept,salary,years
0,1,Alice,Engineering,95000,4
1,2,Bob,Marketing,72000,2
2,3,Carol,Engineering,110000,7
3,4,David,Marketing,68000,1
4,5,Eve,Engineering,102000,5
5,6,Frank,HR,61000,3
6,7,Grace,HR,64000,6
7,8,Hank,Marketing,75000,4


In [3]:
# Select specific columns
query = """
SELECT name, dept, salary
FROM employees
"""
pd.read_sql(query, conn)

,name,dept,salary
0,Alice,Engineering,95000
1,Bob,Marketing,72000
2,Carol,Engineering,110000
3,David,Marketing,68000
4,Eve,Engineering,102000
5,Frank,HR,61000
6,Grace,HR,64000
7,Hank,Marketing,75000


### 1.3 WHERE
`WHERE` filters rows based on a condition.

In [4]:
# Single condition
query = """
SELECT name, salary
FROM employees
WHERE dept = 'Engineering'
"""
pd.read_sql(query, conn)

,name,salary
0,Alice,95000
1,Carol,110000
2,Eve,102000


In [5]:
# Multiple conditions
query = """
SELECT name, dept, salary
FROM employees
WHERE dept = 'Engineering' AND salary > 100000
"""
pd.read_sql(query, conn)

,name,dept,salary
0,Carol,Engineering,110000
1,Eve,Engineering,102000


### 1.4 ORDER BY & LIMIT
`ORDER BY` sorts results. `LIMIT` caps the number of rows returned — useful for previewing large tables.

In [6]:
# Sort by salary descending, return top 3
query = """
SELECT name, dept, salary
FROM employees
ORDER BY salary DESC
LIMIT 3
"""
pd.read_sql(query, conn)

,name,dept,salary
0,Carol,Engineering,110000
1,Eve,Engineering,102000
2,Alice,Engineering,95000


### 1.5 Aggregations
Aggregation functions collapse multiple rows into a single value. Common ones: `COUNT`, `SUM`, `AVG`, `MAX`, `MIN`.

In [7]:
query = """
SELECT
    COUNT(*)    AS total_employees,
    AVG(salary) AS avg_salary,
    MAX(salary) AS max_salary,
    MIN(salary) AS min_salary,
    SUM(salary) AS total_payroll
FROM employees
"""
pd.read_sql(query, conn)

,total_employees,avg_salary,max_salary,min_salary,total_payroll
0,8,80875.0,110000,61000,647000


### 1.6 GROUP BY & HAVING
`GROUP BY` groups rows by a column before aggregating. `HAVING` filters the grouped results — like `WHERE` but applied after aggregation.

In [8]:
# Average salary by department
query = """
SELECT dept, COUNT(*) AS headcount, AVG(salary) AS avg_salary
FROM employees
GROUP BY dept
ORDER BY avg_salary DESC
"""
pd.read_sql(query, conn)

,dept,headcount,avg_salary
0,Engineering,3,102333.333333
1,Marketing,3,71666.666667
2,HR,2,62500.000000


In [9]:
# HAVING — only show depts with avg salary above 70k
query = """
SELECT dept, AVG(salary) AS avg_salary
FROM employees
GROUP BY dept
HAVING avg_salary > 70000
"""
pd.read_sql(query, conn)

,dept,avg_salary
0,Engineering,102333.333333
1,Marketing,71666.666667


---
## Section 2: Joins

### 2.1 INNER JOIN
`INNER JOIN` returns only rows that have a match in both tables.

In [10]:
# Join employees with their department info
query = """
SELECT e.name, e.salary, d.manager, d.budget
FROM employees e
INNER JOIN departments d ON e.dept = d.dept
"""
pd.read_sql(query, conn)

,name,salary,manager,budget
0,Alice,95000,Carol,500000
1,Bob,72000,Hank,200000
2,Carol,110000,Carol,500000
3,David,68000,Hank,200000
4,Eve,102000,Carol,500000
5,Frank,61000,Grace,150000
6,Grace,64000,Grace,150000
7,Hank,75000,Hank,200000


### 2.2 LEFT JOIN
`LEFT JOIN` returns all rows from the left table, with `NULL` for columns from the right table when there's no match.

In [11]:
# All departments, even those with no employees
query = """
SELECT d.dept, d.manager, e.name, e.salary
FROM departments d
LEFT JOIN employees e ON d.dept = e.dept
ORDER BY d.dept
"""
pd.read_sql(query, conn)
# Finance appears with NULL for name and salary — no matching employees

,dept,manager,name,salary
0,Engineering,Carol,Alice,95000.0
1,Engineering,Carol,Carol,110000.0
2,Engineering,Carol,Eve,102000.0
3,Finance,Ivan,NaN,NaN
4,HR,Grace,Frank,61000.0
5,HR,Grace,Grace,64000.0
6,Marketing,Hank,Bob,72000.0
7,Marketing,Hank,David,68000.0
8,Marketing,Hank,Hank,75000.0


### 2.3 When to Use Which
- Use `INNER JOIN` when you only want rows with matches on both sides — e.g. employees who belong to a known department.
- Use `LEFT JOIN` when you want to keep all rows from the left table regardless of matches — e.g. all departments, even ones with no employees.
- In DS, `LEFT JOIN` is more common: you typically start from a base table and enrich it with additional data, accepting `NULL` where data is missing.

---
## Section 3: Practitioner-Level SQL

### 3.1 CTEs (WITH)
A **CTE** (Common Table Expression) is a named temporary result set defined with `WITH`. It makes complex queries readable by breaking them into named steps — instead of nesting subqueries inside each other.

In [12]:
# Find employees who earn above their department's average salary
query = """
WITH dept_avg AS (
    SELECT dept, AVG(salary) AS avg_salary
    FROM employees
    GROUP BY dept
)
SELECT e.name, e.dept, e.salary, ROUND(d.avg_salary, 0) AS dept_avg
FROM employees e
JOIN dept_avg d ON e.dept = d.dept
WHERE e.salary > d.avg_salary
ORDER BY e.dept
"""
pd.read_sql(query, conn)

,name,dept,salary,dept_avg
0,Carol,Engineering,110000,102333.0
1,Grace,HR,64000,62500.0
2,Bob,Marketing,72000,71667.0
3,Hank,Marketing,75000,71667.0


### 3.2 Window Functions
**Window functions** perform calculations across rows related to the current row — without collapsing them like `GROUP BY` does. Each row keeps its identity while gaining a computed value across a window of rows.

Common ones: `ROW_NUMBER`, `RANK`, `LAG`, `LEAD`.

In [13]:
# Rank employees by salary within each department
query = """
SELECT
    name,
    dept,
    salary,
    ROW_NUMBER() OVER (PARTITION BY dept ORDER BY salary DESC) AS rank_in_dept
FROM employees
ORDER BY dept, rank_in_dept
"""
pd.read_sql(query, conn)

,name,dept,salary,rank_in_dept
0,Carol,Engineering,110000,1
1,Eve,Engineering,102000,2
2,Alice,Engineering,95000,3
3,Grace,HR,64000,1
4,Frank,HR,61000,2
5,Hank,Marketing,75000,1
6,Bob,Marketing,72000,2
7,David,Marketing,68000,3


In [14]:
# LAG — compare each employee's salary to the previous row's within the same dept
query = """
SELECT
    name,
    dept,
    salary,
    LAG(salary) OVER (PARTITION BY dept ORDER BY salary DESC) AS prev_salary,
    salary - LAG(salary) OVER (PARTITION BY dept ORDER BY salary DESC) AS diff
FROM employees
ORDER BY dept, salary DESC
"""
pd.read_sql(query, conn)

,name,dept,salary,prev_salary,diff
0,Carol,Engineering,110000,NaN,NaN
1,Eve,Engineering,102000,110000.0,-8000.0
2,Alice,Engineering,95000,102000.0,-7000.0
3,Grace,HR,64000,NaN,NaN
4,Frank,HR,61000,64000.0,-3000.0
5,Hank,Marketing,75000,NaN,NaN
6,Bob,Marketing,72000,75000.0,-3000.0
7,David,Marketing,68000,72000.0,-4000.0


### 3.3 Subqueries
A **subquery** is a query nested inside another query. CTEs are generally preferred for readability, but subqueries are common and worth knowing.

In [15]:
# Subquery in WHERE — employees earning above company average
query = """
SELECT name, dept, salary
FROM employees
WHERE salary > (SELECT AVG(salary) FROM employees)
ORDER BY salary DESC
"""
pd.read_sql(query, conn)

,name,dept,salary
0,Carol,Engineering,110000
1,Eve,Engineering,102000
2,Alice,Engineering,95000


### 3.4 Common Mistakes
- **NULL comparisons**: use `IS NULL` / `IS NOT NULL`, never `= NULL` — `= NULL` always returns false in SQL
- **COUNT(\*) vs COUNT(column)**: `COUNT(*)` counts all rows; `COUNT(column)` ignores NULLs in that column
- **WHERE vs HAVING**: `WHERE` filters before aggregation, `HAVING` filters after — you cannot use aggregate functions in `WHERE`
- **Order of execution**: `FROM` → `WHERE` → `GROUP BY` → `HAVING` → `SELECT` → `ORDER BY` → `LIMIT`

In [16]:
# IS NULL — find departments with no employees
# Finance has no matching rows in employees, so e.name is NULL after LEFT JOIN
query = """
SELECT d.dept, e.name
FROM departments d
LEFT JOIN employees e ON d.dept = e.dept
WHERE e.name IS NULL
"""
pd.read_sql(query, conn)  # returns Finance

,dept,name
0,Finance,None


In [17]:
# COUNT(*) vs COUNT(column)
query = """
SELECT
    COUNT(*)       AS count_all_rows,
    COUNT(salary)  AS count_non_null_salary
FROM employees
"""
pd.read_sql(query, conn)

,count_all_rows,count_non_null_salary
0,8,8
